In [1]:
from datasets import load_dataset

ds = load_dataset("Falah/Alzheimer_MRI")

C:\Users\spicy\AppData\Local\pypoetry\Cache\virtualenvs\finalproject-I8LsYQ7W-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

METRICS = ["accuracy", "f1_macro", "balanced_accuracy"]

In [3]:
import pyhealth
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

class PyHealthAlzheimersCNN(nn.Module):
    def __init__(self, num_classes=4):
        """
        PyHealth BaseModels require dataset, feature_keys, and label_key to automatically
        map your dataloader outputs to the forward function arguments.
        """
        super().__init__()

        self.mode = "multiclass"

        init_channels = 32
        self.num_classes = num_classes # Non, Very Mild, Mild, Moderate

        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=1, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=5, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, 128),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, self.num_classes)
        )

    def forward(self, **kwargs):
        """
        PyHealth passes batches as kwargs using your feature_keys and label_key.
        """
        # 1. Extract the tensors from the PyHealth batch dictionary
        x = kwargs["image"] # The 128x128 images
        labels = kwargs["label"]  # The ground truth labels

        device = next(self.parameters()).device
        x = x.to(device)
        labels = labels.to(device)

        # Forward pass through CNN
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        # Flatten for the MLP
        x = torch.flatten(x, 1)

        # Forward pass MLP
        logits = self.classifier(x)

        # Calculate loss and probabilities
        loss = F.cross_entropy(logits, labels)
        y_prob = F.softmax(logits, dim=1)

        # PyHealth requires the dictionary format
        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels
        }

# Create Sample Dataset in PyHealth

In [4]:
from torchvision import transforms
from pyhealth.trainer import Trainer
from torch.utils.data import Dataset

# dataset is called "ds"

# Create transforms
# Guarantee every image is exactly 1-channel Grayscale and 128x128
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor() # Converts PIL to tensor and scales to [0,1]
])

class PyHealthAlzheimersDataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    item = self.dataset[idx]

    tensor_image = transform(item['image'])

    return {
        'image': tensor_image,
        'label': item['label']
    }


# Convert Train and Test Splits
print("Wrapping Train and Test splits...")
train_dataset = PyHealthAlzheimersDataset(ds['train'])
test_dataset = PyHealthAlzheimersDataset(ds['test'])


print(f"Size of train dataset: {len(train_dataset)}")
print(f"Size of test dataset: {len(test_dataset)}")


Wrapping Train and Test splits...
Size of train dataset: 5120
Size of test dataset: 1280


# Instantiate and use PyHealth Trainer

In [5]:
from torch.utils.data import DataLoader

# Init model
model = PyHealthAlzheimersCNN(
    num_classes=4
)

#Num workers is 0 because I am running on MacOS, was previously 4
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

PyHealthAlzheimersCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dila

Epoch 0 / 50: 100%|██████████| 160/160 [00:35<00:00,  4.50it/s]

--- Train epoch-0, step-160 ---
loss: 1.1072



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 11.51it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
balanced_accuracy: 0.2500
loss: 1.0392



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.8539
f1_macro: 0.6484
balanced_accuracy: 0.6622
loss: 0.3814




Epoch 49 / 50: 100%|██████████| 160/160 [00:34<00:00,  4.62it/s]

--- Train epoch-49, step-8000 ---
loss: 0.1200



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 11.97it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.9148
f1_macro: 0.6717
balanced_accuracy: 0.6573
loss: 0.2722


# Abilation number one per the proposal

In [6]:
class PyHealthAlzheimersCNN_NormVariant(nn.Module):
    """
    Extension 1: Swappable Normalization.
    Supports 'instance', 'group', and 'layer' normalization.
    The original paper uses InstanceNorm2d. GroupNorm and LayerNorm
    have shown benefits on small datasets by avoiding batch-size dependence.
    """
    def __init__(self, num_classes=4, norm_type='instance', num_groups=8):
        super().__init__()

        self.mode = "multiclass"
        self.num_classes = num_classes
        init_channels = 32

        def get_norm(channels, spatial_size=None):
            if norm_type == 'instance':
                return nn.InstanceNorm2d(channels)
            elif norm_type == 'group':
                # num_groups must divide channels evenly
                groups = min(num_groups, channels)
                while channels % groups != 0:
                    groups //= 2
                return nn.GroupNorm(groups, channels)
            elif norm_type == 'layer':
                # LayerNorm over (C, H, W); spatial_size required at build time
                return nn.GroupNorm(1, channels)  # GroupNorm(1, C) == LayerNorm over C dim
            else:
                raise ValueError(f"Unknown norm_type: {norm_type}")

        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=1, stride=1, padding=1),
            get_norm(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            get_norm(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=5, stride=1, padding=1),
            get_norm(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            get_norm(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, 128),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, self.num_classes)
        )

    def forward(self, **kwargs):
        x = kwargs["image"]
        labels = kwargs["label"]

        device = next(self.parameters()).device
        x = x.to(device)
        labels = labels.to(device)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        loss = F.cross_entropy(logits, labels)
        y_prob = F.softmax(logits, dim=1)

        return {"loss": loss, "y_prob": y_prob, "y_true": labels}

In [7]:
model_norm_instance = PyHealthAlzheimersCNN_NormVariant(norm_type='instance')
model_norm_group    = PyHealthAlzheimersCNN_NormVariant(norm_type='group', num_groups=8)
model_norm_layer    = PyHealthAlzheimersCNN_NormVariant(norm_type='layer')

In [8]:
from torch.utils.data import DataLoader

# Init model — swap norm_type to 'instance', 'group', or 'layer'
model = PyHealthAlzheimersCNN_NormVariant(
    num_classes=4,
    norm_type='group',   # change to 'instance' or 'layer' to compare
    num_groups=8
)
# Num workers is 0 on Mac, but was previously 4
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

PyHealthAlzheimersCNN_NormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), paddin

Epoch 0 / 50: 100%|██████████| 160/160 [00:29<00:00,  5.35it/s]

--- Train epoch-0, step-160 ---
loss: 1.0704



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 13.28it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
balanced_accuracy: 0.2500
loss: 1.0391



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.6000
f1_macro: 0.3790
balanced_accuracy: 0.3756
loss: 0.8174




Epoch 49 / 50: 100%|██████████| 160/160 [00:29<00:00,  5.37it/s]

--- Train epoch-49, step-8000 ---
loss: 0.7899



Evaluation: 100%|██████████| 40/40 [00:03<00:00, 13.30it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.5977
f1_macro: 0.3620
balanced_accuracy: 0.3629
loss: 0.8121


# Abilation Number 2

In [9]:
class DepthwiseSeparableConv3d(nn.Module):
    """
    3D Depthwise Separable Convolution block.
    Replaces a standard Conv3d(in_ch, out_ch, k) with:
      1) Depthwise:  Conv3d(in_ch, in_ch, k, groups=in_ch)  — spatial filtering per channel
      2) Pointwise: Conv3d(in_ch, out_ch, 1)               — channel mixing
    Parameter reduction: ~(k^3) x fewer params vs standard Conv3d.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels   # one filter per input channel
        )
        self.pointwise = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=1        # 1x1x1 — mixes channels without touching spatial dims
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class PyHealthAlzheimersCNN_DepthwiseSep3D(nn.Module):
    """
    Extension 2: 3D Depthwise Separable Convolutions.
    The 2D MRI slices are unsqueezed to (B, 1, 1, H, W) — a 3D volume with depth=1.
    Each block uses DepthwiseSeparableConv3d instead of Conv2d, preserving the
    model's ability to reason about 3D structure while cutting parameter count ~9x
    per layer (for kernel_size=3).
    """
    def __init__(self, num_classes=4):
        super().__init__()

        self.mode = "multiclass"
        self.num_classes = num_classes
        init_channels = 32

        self.block1 = nn.Sequential(
            DepthwiseSeparableConv3d(1, init_channels, kernel_size=1, padding=1),
            nn.InstanceNorm3d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))  # only pool H,W; depth=1
        )

        self.block2 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels, init_channels * 2, kernel_size=3, padding=1),
            nn.InstanceNorm3d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.block3 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels * 2, init_channels * 4, kernel_size=5, padding=1),
            nn.InstanceNorm3d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.block4 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels * 4, init_channels * 8, kernel_size=3, padding=1),
            nn.InstanceNorm3d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))   # collapse D, H, W all to 1
        )

        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, 128),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, self.num_classes)
        )

    def forward(self, **kwargs):
        x = kwargs["image"]
        labels = kwargs["label"]

        device = next(self.parameters()).device
        x = x.to(device)
        labels = labels.to(device)

        # (B, 1, H, W) -> (B, 1, 1, H, W): add a depth dimension of size 1
        x = x.unsqueeze(2)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        loss = F.cross_entropy(logits, labels)
        y_prob = F.softmax(logits, dim=1)

        return {"loss": loss, "y_prob": y_prob, "y_true": labels}

In [10]:
from torch.utils.data import DataLoader

# Init model
model = PyHealthAlzheimersCNN_DepthwiseSep3D(
    num_classes=4
)

#Workers = 0 on MacOS, 4 on CoLab
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

PyHealthAlzheimersCNN_DepthwiseSep3D(
  (block1): Sequential(
    (0): DepthwiseSeparableConv3d(
      (depthwise): Conv3d(1, 1, kernel_size=(1, 1, 1), stride=(1, 1, 1), padding=(1, 1, 1))
      (pointwise): Conv3d(1, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    )
    (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): DepthwiseSeparableConv3d(
      (depthwise): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=32)
      (pointwise): Conv3d(32, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    )
    (1): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, cei

Epoch 0 / 50: 100%|██████████| 160/160 [02:54<00:00,  1.09s/it]

--- Train epoch-0, step-160 ---
loss: 1.1014



Evaluation: 100%|██████████| 40/40 [00:06<00:00,  5.98it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
balanced_accuracy: 0.2500
loss: 1.0394



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.5992
f1_macro: 0.3640
balanced_accuracy: 0.3626
loss: 0.8386




Epoch 49 / 50: 100%|██████████| 160/160 [02:53<00:00,  1.09s/it]

--- Train epoch-49, step-8000 ---
loss: 0.7803



Evaluation: 100%|██████████| 40/40 [00:06<00:00,  6.00it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.5953
f1_macro: 0.3816
balanced_accuracy: 0.3793
loss: 0.8260


# Abilation Number 3

In [11]:
class PatchEmbedding(nn.Module):
    """
    Converts a CNN feature map (B, C, H, W) into a sequence of patch tokens
    for the Transformer encoder.
      - Splits the feature map into (patch_size x patch_size) patches
      - Projects each flattened patch to embed_dim via a linear layer
      - Prepends a learnable [CLS] token
      - Adds learnable positional embeddings
    """
    def __init__(self, in_channels, patch_size, embed_dim, feature_map_size):
        super().__init__()
        self.patch_size = patch_size
        num_patches = (feature_map_size // patch_size) ** 2

        # Each patch is a (C * patch_size * patch_size) dimensional vector
        patch_dim = in_channels * patch_size * patch_size
        self.projection = nn.Linear(patch_dim, embed_dim)

        # Learnable [CLS] token — one per batch item
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embeddings — one per patch + one for [CLS]
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size

        # Reshape (B, C, H, W) → (B, num_patches, C * p * p)
        x = x.unfold(2, p, p).unfold(3, p, p)
        # x shape after unfold: (B, C, H//p, W//p, p, p)
        x = x.contiguous().view(B, C, -1, p * p)
        # x: (B, C, num_patches, p*p)
        x = x.permute(0, 2, 1, 3)
        # x: (B, num_patches, C, p*p)
        x = x.contiguous().view(B, -1, C * p * p)
        # x: (B, num_patches, C*p*p)

        # Project each patch to embed_dim
        x = self.projection(x)
        # x: (B, num_patches, embed_dim)

        # Prepend [CLS] token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        # x: (B, num_patches + 1, embed_dim)

        # Add positional embeddings
        x = x + self.pos_embedding

        return x


class PyHealthAlzheimersCNN_ViT(nn.Module):
    """
    Extension 3: CNN + Vision Transformer Hybrid.
    Architecture:
      [Input 128x128]
           |
      block1 (Conv2d) → 64x64
      block2 (Conv2d) → 32x32      ← local feature extraction
           |
      PatchEmbedding (patch_size=4) → (B, 64+1, embed_dim)
           |
      Transformer Encoder (N layers) ← global context modeling
           |
      [CLS] token → classifier head
    """
    def __init__(self, num_classes=4, embed_dim=128, num_heads=4,
                 num_transformer_layers=4, dropout=0.1):
        super().__init__()

        self.mode = "multiclass"
        self.num_classes = num_classes
        init_channels = 32

        # --- CNN backbone: local feature extraction (2 blocks → 32x32 feature map) ---
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=1, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)          # 128 → 64
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)          # 64 → 32
        )

        # --- Patch Embedding: split 32x32 feature map into 4x4 patches → 64 patches ---
        cnn_out_channels = init_channels * 2   # 64
        feature_map_size = 32                  # after 2 pooling layers on 128x128 input
        patch_size = 4                         # 32 / 4 = 8 patches per dim → 64 patches total

        self.patch_embed = PatchEmbedding(
            in_channels=cnn_out_channels,
            patch_size=patch_size,
            embed_dim=embed_dim,
            feature_map_size=feature_map_size
        )

        # --- Transformer Encoder: global context over patch sequence ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,     # standard 4x expansion in FFN
            dropout=dropout,
            batch_first=True                   # (B, seq, embed_dim) convention
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers
        )

        # --- Classifier: operates on the [CLS] token output ---
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, self.num_classes)
        )

    def forward(self, **kwargs):
        x = kwargs["image"]
        labels = kwargs["label"]

        device = next(self.parameters()).device
        x = x.to(device)
        labels = labels.to(device)

        # Step 1: CNN local feature extraction
        x = self.block1(x)                     # (B, 32, 64, 64)
        x = self.block2(x)                     # (B, 64, 32, 32)

        # Step 2: Patch tokenization
        x = self.patch_embed(x)                # (B, 65, embed_dim)  [64 patches + CLS]

        # Step 3: Transformer global context modeling
        x = self.transformer(x)                # (B, 65, embed_dim)

        # Step 4: Extract [CLS] token (index 0) for classification
        cls_output = x[:, 0, :]                # (B, embed_dim)

        # Step 5: Classify
        logits = self.classifier(cls_output)   # (B, num_classes)

        loss = F.cross_entropy(logits, labels)
        y_prob = F.softmax(logits, dim=1)

        return {"loss": loss, "y_prob": y_prob, "y_true": labels}

In [12]:
from torch.utils.data import DataLoader

# Init model
model = PyHealthAlzheimersCNN_ViT(
    num_classes=4,
    embed_dim=128,
    num_heads=4,
    num_transformer_layers=4,
    dropout=0.1
)

#Workers is 0 on MacOS, 4 on CoLab
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

PyHealthAlzheimersCNN_ViT(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (patch_embed): PatchEmbedding(
    (projection): Linear(in_features=1024, out_features=128, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLi

Epoch 0 / 50: 100%|██████████| 160/160 [00:42<00:00,  3.80it/s]

--- Train epoch-0, step-160 ---
loss: 1.0781



Evaluation: 100%|██████████| 40/40 [00:02<00:00, 15.18it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
balanced_accuracy: 0.2500
loss: 1.0369



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.9047
f1_macro: 0.8857
balanced_accuracy: 0.9019
loss: 0.3526




Epoch 49 / 50: 100%|██████████| 160/160 [00:41<00:00,  3.83it/s]

--- Train epoch-49, step-8000 ---
loss: 0.0247



Evaluation: 100%|██████████| 40/40 [00:02<00:00, 15.30it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.8914
f1_macro: 0.8643
balanced_accuracy: 0.8642
loss: 0.4905
